In [ ]:
import importlib
import numpy as np
import seaborn as sns

import matplotlib.pyplot as plt

# adjust path
path_to_hkpt = '../'
import sys
sys.path.append(path_to_hkpt)

import hk_parallel_transport as hkpt
hkpt = importlib.reload(hkpt)

plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman"],
})


In [ ]:
n_1 = 1000
n_2 = 2000

mean_1 = np.array([0.0])
cov_1 = np.array([[1.0]])
mean_2 = np.array([10.0])
cov_2 = np.array([[1.0]])

data_1 = np.random.multivariate_normal(mean_1, cov_1, n_1)
data_2 = np.random.multivariate_normal(mean_2, cov_2, n_2)
# scale everything so that it is at distance <= pi/4
max_val = max(np.max(np.abs(data_1)), np.max(np.abs(data_2)))
min_val = min(np.min(np.abs(data_1)), np.min(np.abs(data_2)))
scale_factor = (np.pi / 4) / max_val
data_1 *= scale_factor
data_2 *= scale_factor

# unnormalized histogram of the data, smooth to get continuous curve
plt.figure(figsize=(10, 6))
# sns.histplot(data_1, bins=20, element='poly')
# sns.histplot(data_2, bins=20, element='poly')
plt.hist(data_1, bins=20,  alpha=0.5, label='Measure 1')
plt.hist(data_2, bins=20, alpha=0.5, label='Measure 2')
# no legend
plt.legend().set_visible(False)

# no axes
plt.axis('off')
plt.show()

In [ ]:
# empirical (unnormalized) measures
mu1 = hkpt.EmpiricalMeasure(data_1, weights=np.ones(n_1))
mu2 = hkpt.EmpiricalMeasure(data_2, weights=np.ones(n_2))

# obtain cone interpolations
lambdas, radiis = hkpt.let_lift(mu1, mu2, N=5)
# visualize the cone interpolation in polar coordinates


In [ ]:
# plot cone heatmap over time in 5 subplots
time_ids = np.linspace(0, len(lambdas) - 1, 5, dtype=int)
embedded_x = []
embedded_y = []
for idx in time_ids:
    cone_measure = lambdas[idx]
    embedded_x.append(cone_measure.radii * np.cos(cone_measure.samples[:, 0]))
    embedded_y.append(cone_measure.radii * np.sin(cone_measure.samples[:, 0]))

x_lim = (min(arr.min() for arr in embedded_x) - 0.1, max(arr.max() for arr in embedded_x) + 0.1)
y_lim = (min(arr.min() for arr in embedded_y) - 0.1, max(arr.max() for arr in embedded_y) + 0.1)

fig, axes = plt.subplots(1, 5, figsize=(18, 3.6), constrained_layout=True)
for ax, idx, x_embed, y_embed in zip(axes, time_ids, embedded_x, embedded_y):
    cone_measure = lambdas[idx]
    weights = cone_measure.weights / cone_measure.weights.sum()  # normalize weights for better visualization
    hb = ax.hexbin(
        x_embed,
        y_embed,
        C=weights,
        reduce_C_function=np.sum,
        gridsize=32,
        mincnt=1,
        cmap="mako",
    )
    ax.set_title(rf"$t={idx / (len(lambdas) - 1):.2f}$")
    ax.set_xlim(*x_lim)
    ax.set_ylim(*y_lim)
    ax.set_aspect("equal")
    ax.axis("off")
    # plot concentric circles for reference
    for r in np.linspace(0.5, 2.5, 5):
        ax.plot(r * np.cos(np.linspace(0, 2 * np.pi, 100)), r * np.sin(np.linspace(0, 2 * np.pi, 100)), color="gray", linestyle="--", linewidth=0.5)

fig.colorbar(hb, ax=axes, shrink=0.8, label="cone mass")
fig.suptitle(r"Cone interpolation viewed in polar coordinates", y=1.15, fontsize=20)
# plt.savefig('../Figures/HK_cone_interpolation_polar.pdf', dpi=800)
plt.show()

In [ ]:
# plot projected measures over time in 5 subplots
projected_measures = [hkpt.project_cone_measure(lambdas[idx]) for idx in time_ids]
all_projected_x = np.concatenate([measure.samples[:, 0] for measure in projected_measures])
bins = np.linspace(all_projected_x.min() - 0.5, all_projected_x.max() + 0.5, 45)

fig, axes = plt.subplots(1, 5, figsize=(18, 3.6), sharey=True, constrained_layout=True)
for ax, idx, measure in zip(axes, time_ids, projected_measures):
    ax.hist(measure.samples[:, 0], bins=bins, weights=measure.weights, color="tab:blue", alpha=0.65)
    ax.set_title(rf"$t={idx / (len(lambdas) - 1):.2f}$" + "\n" + rf"total mass $={measure.total_mass():.0f}$")
    ax.set_xlabel(r"$x$")

axes[0].set_ylabel("projected mass")
fig.suptitle(r"Projected empirical HK measures along the interpolation", y=1.1, fontsize=20)
# plt.savefig('../Figures/HK_cone_interpolation_projected.pdf', dpi=800)
plt.show()

In [ ]:
## Sanity check: align projected measure samples to original measure samples and visualize

indices = hkpt.align_samples_to_support(
    projected_measures[-1].samples,
    mu2.samples
)
plt.scatter(projected_measures[-1].samples[:, 0], np.zeros_like(projected_measures[-1].samples[:, 0]), color='tab:orange', alpha=0.5, label='Projected measure samples')
plt.scatter(mu2.samples[:, 0], np.ones_like(mu2.samples[:, 0]), color='tab:blue', alpha=0.5, label='Original measure samples')

In [ ]:
################################# Isometric lift #################################

# recursive isometric lift built from the intermediate base measures
iso_lambdas, iso_radiis = hkpt.isometric_lift(
    mu1, 
    mu2, 
    N=2,
    radial_aggregation_mode='mean_radius'
)
iso_time_ids = np.linspace(0, len(iso_lambdas) - 1, 5, dtype=int)
# visualize the recursive isometric lift in polar coordinates


In [ ]:
# plot recursive isometric-lift cone heatmap over time in 5 subplots
iso_embedded_x = []
iso_embedded_y = []
for idx in iso_time_ids:
    cone_measure = iso_lambdas[idx]
    iso_embedded_x.append(cone_measure.radii * np.cos(cone_measure.samples[:, 0]))
    iso_embedded_y.append(cone_measure.radii * np.sin(cone_measure.samples[:, 0]))

iso_x_lim = (min(arr.min() for arr in iso_embedded_x) - 0.1, max(arr.max() for arr in iso_embedded_x) + 0.1)
iso_y_lim = (min(arr.min() for arr in iso_embedded_y) - 0.1, max(arr.max() for arr in iso_embedded_y) + 0.1)

fig, axes = plt.subplots(1, 5, figsize=(18, 3.6), constrained_layout=True)
for ax, idx, x_embed, y_embed in zip(axes, iso_time_ids, iso_embedded_x, iso_embedded_y):
    cone_measure = iso_lambdas[idx]
    weights = cone_measure.weights / cone_measure.weights.sum()  # normalize weights for better visualization
    hb = ax.hexbin(
        x_embed,
        y_embed,
        C=weights,
        reduce_C_function=np.sum,
        gridsize=32,
        mincnt=1,
        cmap="rocket",
    )
    ax.set_title(rf"$t={idx / (len(iso_lambdas) - 1):.2f}$")
    ax.set_xlim(*iso_x_lim)
    ax.set_ylim(*iso_y_lim)
    ax.set_aspect("equal")
    ax.axis("off")
    # plot concentric circles for reference
    for r in np.linspace(0.5, 2.5, 5):
         ax.plot(r * np.cos(np.linspace(0, 2 * np.pi, 100)), r * np.sin(np.linspace(0, 2 * np.pi, 100)), color="gray", linestyle="--", linewidth=0.5)

fig.colorbar(hb, ax=axes, shrink=0.8, label="cone mass")
fig.suptitle(r"Isometric lift in polar coordinates", y=1.1, fontsize=20)
plt.savefig('../Figures/HK_cone_interpolation_recursive.pdf', dpi=800)
plt.show()

In [ ]:
# plot projected measures from the recursive isometric lift over time in 5 subplots
iso_projected_measures = [hkpt.project_cone_measure(iso_lambdas[idx]) for idx in iso_time_ids]
iso_all_projected_x = np.concatenate([measure.samples[:, 0] for measure in iso_projected_measures])
iso_bins = np.linspace(iso_all_projected_x.min() - 0.5, iso_all_projected_x.max() + 0.5, 45)

fig, axes = plt.subplots(1, 5, figsize=(18, 3.6), sharey=True, constrained_layout=True)
for ax, idx, measure in zip(axes, iso_time_ids, iso_projected_measures):
    ax.hist(measure.samples[:, 0], bins=iso_bins, weights=measure.weights, color="tab:orange", alpha=0.65)
    ax.set_title(rf"$t={idx / (len(iso_lambdas) - 1):.2f}$" + "\n" + rf"total mass $={measure.total_mass():.0f}$")
    ax.set_xlabel(r"$x$")

axes[0].set_ylabel("projected mass")
fig.suptitle(r"Interpolated HK measures from the isometric lift", y=1.1, fontsize=20)
plt.savefig('../Figures/HK_cone_interpolation_recursive_projected.pdf', dpi=800)
plt.show()

In [ ]:
## Sanity check: align projected measure samples to original measure samples and visualize

try:
    indices = hkpt.align_samples_to_support(
        iso_projected_measures[-1].samples,
        mu2.samples
    )
except Exception as e:
    print(f"Alignment failed with error: {e}")

plt.scatter(iso_projected_measures[-1].samples[:, 0], np.zeros_like(iso_projected_measures[-1].samples[:, 0]), color='tab:orange', alpha=0.5, label='Projected measure samples')
plt.scatter(mu2.samples[:, 0], np.ones_like(mu2.samples[:, 0]), color='tab:blue', alpha=0.5, label='Original measure samples')